#  ⭐ `dim_dose`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root, normalize_dose

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet")  
bronze['DOSE']= bronze['DOSE'].fillna('DESCONHECIDO')
bronze = (
    bronze[['DOSE']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['DOSE', 'FREQUENCIA_REGISTROS'], dtype='object')

In [4]:
bronze.DOSE.nunique()

9764

In [5]:
bronze.head(40)

,DOSE,FREQUENCIA_REGISTROS
0,DESCONHECIDO,339554
1,100 milligram (mg),10204
2,50 milligram (mg),9612
3,10 milligram (mg),8231
4,300 milligram (mg),7323
5,1 dosage form ({DF}),6812
6,20 milligram (mg),6795
7,40 milligram (mg),6051
8,500 milligram (mg),5743
9,1 gram (g),5266


In [6]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_DOSE.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [7]:
silver = normalize_dose(bronze, col="DOSE")

silver.columns

Index(['DOSE', 'FREQUENCIA_REGISTROS', 'DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR',
       'DOSE_VALOR'],
      dtype='object')

In [8]:
silver[['DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR']].drop_duplicates().sort_values(by='DOSE_TIPO_VALOR')

,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR
0,desconhecido,0
1,milligram (mg),1
29,millilitre (mL),2
9,gram (g),3
31,microgram (ug),4
5,dosage form ({DF}),5
81,milligram per kilogram (mg/kg),6
101,milligram per square metre (mg/m2),7
60,international unit ([iU]),8
84,drop (1/12 millilitre) ([drp]),9


In [9]:
silver.DOSE_VALOR.value_counts(dropna=False).head(10)

DOSE_VALOR
1.0      104
5.0       82
2.0       70
500.0     59
4.0       58
10.0      53
100.0     50
3.0       50
50.0      47
20.0      45
Name: count, dtype: int64

In [10]:
silver.query("DOSE_TIPO_CHAVE == 'desconhecido'").DOSE.value_counts(dropna=False).head(40)

DOSE
DESCONHECIDO                                      1
1.1 microgram per kilogram per hour (ug/kg/h)     1
1 milligram per minute (mg/min)                   1
1 month (mo)                                      1
1 per 250 millilitre (/(250.mL))                  1
1 second (s)                                      1
1 week (wk)                                       1
1.0 millimole (mmol)                              1
1.000                                             1
1 gram per 24 hour (g/(24.h))                     1
1.25 microgram per kilogram (ug/kg)               1
1 millicurie (mCi)                                1
1.250.00                                          1
1.2E+07 dosage form ({DF})                        1
1.3                                               1
1.3 milligram per cubic metre (mg/m3)             1
1.34                                              1
1.1 milligram per 10 hour (mg/(10.h))             1
1.12 microgram per kilogram per hour (ug/kg/h)    1
1.14 mi

In [11]:
hist = silver

In [12]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_dose/hist_dim_dose.parquet", index=False)

In [13]:
hist.to_csv(Path(project_root) / "data/02_silver/hist_dim_dose/hist_dim_dose_MANUAL.csv", sep=";", index=False)

# 🥇 Gold


In [14]:
hist.columns

Index(['DOSE', 'FREQUENCIA_REGISTROS', 'DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR',
       'DOSE_VALOR'],
      dtype='object')

In [15]:
dim = hist[['DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR']].sort_values(by='DOSE_TIPO_VALOR').drop_duplicates()
dim

,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR
0,desconhecido,0
646,milligram (mg),1
3304,millilitre (mL),2
2670,gram (g),3
1929,microgram (ug),4
325,dosage form ({DF}),5
4618,milligram per kilogram (mg/kg),6
5469,milligram per square metre (mg/m2),7
8704,international unit ([iU]),8
84,drop (1/12 millilitre) ([drp]),9


In [16]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_dose/dim_dose.csv",sep=";", index=False)